<a href="https://colab.research.google.com/github/EikESousA/mestrado-paa/blob/main/Semin%C3%A1rio_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Entrada do Problema e Instância

In [1]:
from random import randint
from itertools import permutations, combinations
from time import time

In [2]:
cities = [
    "Amparo de São Francisco", "Aquidabã", "Aracaju", "Arauá", "Areia Branca",
    "Barra dos Coqueiros", "Boquim", "Brejo Grande", "Campo do Brito", "Canhoba",
    "Canindé de São Francisco", "Capela", "Carira", "Carmópolis", "Cedro de São João",
    "Cristinápolis", "Cumbe", "Divina Pastora", "Estância", "Feira Nova",
    "Frei Paulo", "Gararu", "General Maynard", "Gracho Cardoso", "Ilha das Flores",
    "Indiaroba", "Itabaiana", "Itabaianinha", "Itabi", "Japaratuba",
    "Japoatã", "Lagarto", "Laranjeiras", "Macambira", "Malhada dos Bois",
    "Malhador", "Maruim", "Moita Bonita", "Monte Alegre de Sergipe", "Muribeca",
    "Neópolis", "Nossa Senhora Aparecida", "Nossa Senhora da Glória", "Nossa Senhora das Dores", "Nossa Senhora de Lourdes",
    "Nossa Senhora do Socorro", "Pacatuba", "Pedra Mole", "Pedrinhas", "Pinhão",
    "Pirambu", "Poço Redondo", "Poço Verde", "Porto da Folha", "Propriá",
    "Riachão do Dantas", "Riachuelo", "Ribeirópolis", "Rosário do Catete", "Salgado",
    "Santa Luzia do Itanhy", "Santa Rosa de Lima", "Santana do São Francisco", "Santo Amaro das Brotas", "São Cristóvão",
    "São Domingos", "São Francisco", "São Miguel do Aleixo", "Simão Dias", "Siriri",
    "Telha", "Tobias Barreto", "Tomar do Geru", "Umbaúba"
];

def generate_distances(n):
    distances = [[0 if i == j else randint(10, 150) for j in range(n)] for i in range(n)]

    for i in range(n):
        for j in range(i + 1, n):
            distances[j][i] = distances[i][j]

    return distances

In [3]:
n = 5
distances = generate_distances(n)
format_names = [cidade[:5] for cidade in cities[:n]]

print("     ", "  ".join(format_names))
for i in range(n):
    line = f"{format_names[i]:<5} " + "    ".join(f"{distances[i][j]:<3}" for j in range(n))
    print(line)

      Ampar  Aquid  Araca  Arauá  Areia
Ampar 0      55     130    141    19 
Aquid 55     0      92     105    147
Araca 130    92     0      146    122
Arauá 141    105    146    0      100
Areia 19     147    122    100    0  


In [4]:
class BruteForce:
  def __init__(self, cities, distances):
    self.cities = cities
    self.distances = distances
    self.n = len(distances)

    self.path_route = None
    self.distance_route = float('inf')

  def print_route(self):
    route = ''
    route = route + ' -> '.join([cities[i] for i in self.path_route])
    route = route + ' -> ' + cities[0]
    return route

  def calculate_distance(self, route):
    distance = 0
    m = len(route)

    for i in range(m - 1):
        distance += self.distances[route[i]][route[i + 1]]

    distance += self.distances[route[-1]][route[0]]
    return distance

  def execute(self):
    init_time = time()
    permutations_routes = permutations(range(self.n))

    for permutation_route in permutations_routes:
        distance = self.calculate_distance(permutation_route)

        if distance < self.distance_route:
            self.distance_route = distance
            self.path_route = permutation_route

    end_time = time()

    total_time = end_time - init_time

    print("| -------------------------- FORCA BRUTA -------------------------- |")
    print("| Quantidade: ", self.n)
    print("| Rota:", self.print_route())
    print("| Distancia:", self.distance_route)
    print("| Tempo:", total_time)
    print("| ----------------------------------------------------------------- |")

    return total_time

In [5]:
class HeldKarp:
  def __init__(self, cities, distances):
    self.cities = cities
    self.distances = distances
    self.n = len(distances)

    self.paths_routes = {}
    self.distance_route = float('inf')

    self.bitmask = 0
    self.last_node = -1

  def print_route(self):
    best_routes = self.get_best_route()

    route = ''
    route = route + ' -> '.join([self.cities[i] for i in best_routes])
    return route

  def get_best_route(self):
    last_node = self.last_node
    bitmask = self.bitmask

    path = [0]

    while bitmask:
        path.append(last_node)
        bitmask_without_last = bitmask ^ (1 << last_node)

        hasNext = False

        for i in range(1, self.n):
            if bitmask_without_last & (1 << i):
                if self.paths_routes.get((bitmask, last_node)) == self.paths_routes.get((bitmask_without_last, i)) + self.distances[i][last_node]:
                    last_node = i
                    bitmask = bitmask_without_last
                    hasNext = True
                    break

        if not hasNext:
            break

    path.append(0)
    return path[::-1]

  def execute(self):
    init_time = time()

    for i in range(1, self.n):
        self.paths_routes[(1 << i, i)] = self.distances[0][i]

    for subset_second in range(2, self.n):
      for subset_rest in combinations(range(1, self.n), subset_second):
        bitmask = 0

        for bit in subset_rest:
          bitmask |= (1 << bit)

        for i in subset_rest:
          self.paths_routes[(bitmask, i)] = float('inf')
          bitmask_without_i = bitmask ^ (1 << i)

          for j in subset_rest:
            if j == i:
              continue

            self.paths_routes[(bitmask, i)] = min(self.paths_routes[(bitmask, i)],
                                                  self.paths_routes[(bitmask_without_i, j)] +
                                                  self.distances[j][i]
                                                )

    self.bitmask = (1 << self.n) - 2

    for i in range(1, self.n):
        distance =  min(self.distance_route, self.paths_routes[(bitmask, i)] + self.distances[i][0])
        if distance < self.distance_route:
            self.distance_route = distance
            self.last_node = i

    end_time = time()

    total_time = end_time - init_time

    print("| --------------------------- HELD KERP --------------------------- |")
    print("| Quantidade: ", self.n)
    print("| Rota:", self.print_route())
    print("| Distancia:", self.distance_route)
    print("| Tempo:", total_time)
    print("| ----------------------------------------------------------------- |")

    return total_time

In [6]:
for i in range(5, 14):
  distances = generate_distances(i)
  brute_force = BruteForce(cities, distances)
  brute_force.execute()

| -------------------------- FORCA BRUTA -------------------------- |
| Quantidade:  5
| Rota: Amparo de São Francisco -> Aracaju -> Arauá -> Aquidabã -> Areia Branca -> Amparo de São Francisco
| Distancia: 329
| Tempo: 0.0001404285430908203
| ----------------------------------------------------------------- |
| -------------------------- FORCA BRUTA -------------------------- |
| Quantidade:  6
| Rota: Amparo de São Francisco -> Aracaju -> Aquidabã -> Barra dos Coqueiros -> Areia Branca -> Arauá -> Amparo de São Francisco
| Distancia: 274
| Tempo: 0.0013899803161621094
| ----------------------------------------------------------------- |
| -------------------------- FORCA BRUTA -------------------------- |
| Quantidade:  7
| Rota: Amparo de São Francisco -> Areia Branca -> Boquim -> Aquidabã -> Aracaju -> Arauá -> Barra dos Coqueiros -> Amparo de São Francisco
| Distancia: 303
| Tempo: 0.006424903869628906
| ----------------------------------------------------------------- |
| -------

Held-Karp

In [7]:
for i in range(5, 25):
  distances = generate_distances(i)
  held_karp = HeldKarp(cities, distances)
  held_karp.execute()

| --------------------------- HELD KERP --------------------------- |
| Quantidade:  5
| Rota: Amparo de São Francisco -> Areia Branca -> Aquidabã -> Arauá -> Aracaju -> Amparo de São Francisco
| Distancia: 320
| Tempo: 5.53131103515625e-05
| ----------------------------------------------------------------- |
| --------------------------- HELD KERP --------------------------- |
| Quantidade:  6
| Rota: Amparo de São Francisco -> Areia Branca -> Aracaju -> Aquidabã -> Barra dos Coqueiros -> Arauá -> Amparo de São Francisco
| Distancia: 235
| Tempo: 0.00014519691467285156
| ----------------------------------------------------------------- |
| --------------------------- HELD KERP --------------------------- |
| Quantidade:  7
| Rota: Amparo de São Francisco -> Boquim -> Barra dos Coqueiros -> Arauá -> Aquidabã -> Areia Branca -> Aracaju -> Amparo de São Francisco
| Distancia: 330
| Tempo: 0.0003762245178222656
| ----------------------------------------------------------------- |
| ------